# 🚁 Detección de Drones y Personas con YOLOv8
### Proyecto: Detección de objetos para enjambre de drones autónomos

**Clases objetivo:**
- `drone` — Para evitar colisiones con otros agentes del enjambre
- `person` — Obstáculo dinámico para seguridad de vuelo

**Arquitectura:** YOLOv8m (You Only Look Once v8, variante Medium)

**Hardware requerido:** NVIDIA RTX 3070 + CUDA 12.x

---
## 📦 CELDA 1 — Instalación de dependencias
Solo ejecutar una vez. Reiniciar el kernel después si es necesario.

In [ ]:
# Instalar PyTorch con soporte CUDA 12.x (compatible con CUDA 12.9 del driver)
# IMPORTANTE: Ejecutar en terminal de VSC con el entorno virtual activado:
#   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Instalar el resto de dependencias
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', 'albumentations', 'opencv-python',
                'matplotlib', 'pandas', 'numpy', 'pyyaml', 'tqdm', 'Pillow',
                'seaborn', 'scikit-learn'], check=True)

print('✅ Dependencias instaladas correctamente')

---
## ✅ CELDA 2 — Verificación del entorno GPU

In [ ]:
import torch
import subprocess

print('=' * 55)
print('       VERIFICACIÓN DEL ENTORNO DE ENTRENAMIENTO')
print('=' * 55)
print(f'PyTorch version   : {torch.__version__}')
print(f'CUDA disponible   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_free  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1e9
    print(f'GPU detectada     : {gpu_name}')
    print(f'VRAM total        : {vram_total:.1f} GB')
    print(f'VRAM disponible   : {vram_free:.1f} GB')
    print(f'Versión CUDA      : {torch.version.cuda}')
    print('\n🟢 Entrenamiento en GPU activado — RTX 3070 lista')
else:
    print('\n🔴 GPU no detectada. Verifica la instalación de PyTorch+CUDA:')
    print('   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121')

print('=' * 55)

---
## 📁 CELDA 3 — Configuración de rutas de los datasets

### Datasets utilizados y justificación:

| Dataset | Clase usada | Imágenes | Razón de selección |
|---|---|---|---|
| `Drone_Dataset_UAV` | drone | ~1,359 | UAVs rotatorios reales, perspectiva de vuelo |
| `Drone Detection` | drone | ~N/A | Mayor variedad de entornos y condiciones de luz |
| `Person Detection UAV` | person | ~15,500 | Personas vistas desde dron (perspectiva correcta) |

**Total estimado después de filtrado:** ~10,000–18,000 imágenes  
**Partición:** 70% train / 20% valid / 10% test

**¿Por qué estas imágenes son representativas para vuelo de dron?**  
- Todas provienen de perspectiva aérea o semi-aérea real  
- Incluyen variación de altitud, iluminación y fondo  
- Las personas están fotografiadas desde arriba (como las vería el dron)

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURA AQUÍ las rutas donde descomprimiste los datasets
# ============================================================
BASE_DIR = Path('C:/drone_project')   # Carpeta raíz del proyecto

# Rutas de los datasets originales
DATASET_UAV_DIR      = Path('C:/datasets/Drone_Dataset_UAV/drone_dataset_yolo/dataset_txt')
DATASET_DETECT_DIR   = Path('C:/datasets/DroneDetection/drone-detection-new.v5-new-train.yolov8')
DATASET_PERSON_DIR   = Path('C:/datasets/PersonDetectionUAV/PASCAL_VOC_UAQ_MSUAV')

# Carpeta de salida unificada
UNIFIED_DIR          = BASE_DIR / 'unified_dataset'
TRAIN_IMG  = UNIFIED_DIR / 'images' / 'train'
TRAIN_LBL  = UNIFIED_DIR / 'labels' / 'train'
VALID_IMG  = UNIFIED_DIR / 'images' / 'valid'
VALID_LBL  = UNIFIED_DIR / 'labels' / 'valid'
TEST_IMG   = UNIFIED_DIR / 'images' / 'test'
TEST_LBL   = UNIFIED_DIR / 'labels' / 'test'

for folder in [TRAIN_IMG, TRAIN_LBL, VALID_IMG, VALID_LBL, TEST_IMG, TEST_LBL]:
    folder.mkdir(parents=True, exist_ok=True)

print('✅ Estructura de carpetas creada en:', UNIFIED_DIR)
print('\nVerifica que las siguientes rutas existen en tu PC:')
for name, path in [('UAV Dataset', DATASET_UAV_DIR),
                   ('Drone Detection', DATASET_DETECT_DIR),
                   ('Person UAV', DATASET_PERSON_DIR)]:
    status = '✅' if path.exists() else '❌ NO ENCONTRADA'
    print(f'  {status}  {name}: {path}')

---
## 🔀 CELDA 4 — Unificación y remapeo de clases

Los datasets originales tienen diferentes IDs de clase. Los unificamos a:
- **Clase 0** → `drone`
- **Clase 1** → `person`

Los archivos `.txt` de YOLO tienen formato: `class_id cx cy w h` (normalizados)

In [ ]:
import shutil
import random
import os
from pathlib import Path
from tqdm import tqdm

random.seed(42)

def remap_and_copy(img_paths, lbl_paths, class_mapping, split_ratios=(0.7, 0.2, 0.1)):
    """
    Copia imágenes y etiquetas al dataset unificado remapeando IDs de clase.
    class_mapping: dict {id_original: id_nuevo} — clases no en el dict se descartan
    """
    paired = [(i, l) for i, l in zip(img_paths, lbl_paths) if i.exists() and l.exists()]
    random.shuffle(paired)

    n = len(paired)
    n_train = int(n * split_ratios[0])
    n_valid = int(n * split_ratios[1])
    splits = [
        ('train', paired[:n_train]),
        ('valid', paired[n_train:n_train + n_valid]),
        ('test',  paired[n_train + n_valid:])
    ]

    counts = {'train': 0, 'valid': 0, 'test': 0}
    for split_name, items in splits:
        img_dst = UNIFIED_DIR / 'images' / split_name
        lbl_dst = UNIFIED_DIR / 'labels' / split_name
        for img_path, lbl_path in tqdm(items, desc=f'  {split_name}'):
            # Leer etiqueta y remap
            new_lines = []
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    orig_cls = int(parts[0])
                    if orig_cls in class_mapping:
                        new_cls = class_mapping[orig_cls]
                        new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
            if not new_lines:
                continue  # Imagen sin clases útiles, se omite

            # Nombre único para evitar colisiones entre datasets
            stem = img_path.stem
            suffix = img_path.suffix
            new_name = f'{split_name[:2]}_{stem}'

            shutil.copy2(img_path, img_dst / f'{new_name}{suffix}')
            with open(lbl_dst / f'{new_name}.txt', 'w') as f:
                f.write('\n'.join(new_lines))
            counts[split_name] += 1
    return counts


print('🔄 Procesando Dataset 1: Drone_Dataset_UAV (clase drone → 0)...')
# En este dataset, la única clase es dron (id 0 original)
uav_imgs = sorted(DATASET_UAV_DIR.glob('*.jpg')) + sorted(DATASET_UAV_DIR.glob('*.png'))
uav_lbls = [DATASET_UAV_DIR / (p.stem + '.txt') for p in uav_imgs]
c1 = remap_and_copy(uav_imgs, uav_lbls, class_mapping={0: 0})  # drone → 0
print(f'   Imágenes procesadas: {c1}')

print('\n🔄 Procesando Dataset 2: Drone Detection (clase Drone → 0, resto descartado)...')
# Clases: 0=AirPlane, 1=Drone, 2=Helicopter → solo queremos Drone
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DETECT_DIR / split / 'images'
    lbl_dir = DATASET_DETECT_DIR / split / 'labels'
    if not img_dir.exists():
        print(f'   ⚠️  No encontrado: {img_dir}')
        continue
    imgs = sorted(img_dir.glob('*.jpg'))
    lbls = [lbl_dir / (p.stem + '.txt') for p in imgs]
    c = remap_and_copy(imgs, lbls, class_mapping={1: 0})  # Drone(1) → 0
    print(f'   {split}: {c}')

print('\n🔄 Procesando Dataset 3: Person Detection UAV (clase Person → 1)...')
person_img_dir = DATASET_PERSON_DIR / 'images'
person_lbl_dir = DATASET_PERSON_DIR / 'labels'
if person_img_dir.exists():
    p_imgs = sorted(person_img_dir.glob('*.jpg'))
    p_lbls = [person_lbl_dir / (p.stem + '.txt') for p in p_imgs]
    c3 = remap_and_copy(p_imgs, p_lbls, class_mapping={0: 1})  # Person → 1
    print(f'   Imágenes procesadas: {c3}')
else:
    print(f'   ⚠️  No encontrado: {person_img_dir}')

# Conteo final
print('\n📊 RESUMEN DEL DATASET UNIFICADO:')
for split in ['train', 'valid', 'test']:
    n_img = len(list((UNIFIED_DIR / 'images' / split).glob('*')))
    n_lbl = len(list((UNIFIED_DIR / 'labels' / split).glob('*.txt')))
    print(f'  {split:6s}: {n_img:6d} imágenes | {n_lbl:6d} etiquetas')

---
## 📝 CELDA 5 — Crear archivo data.yaml
YOLOv8 necesita este archivo de configuración para saber dónde están los datos y cuántas clases hay.

In [ ]:
import yaml

data_yaml = {
    'path': str(UNIFIED_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/valid',
    'test':  'images/test',
    'nc': 2,
    'names': ['drone', 'person']
}

yaml_path = UNIFIED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print('✅ data.yaml creado en:', yaml_path)
print('\nContenido:')
with open(yaml_path) as f:
    print(f.read())

---
## 🔍 CELDA 6 — Exploración visual del dataset
Antes de entrenar, verificamos que las anotaciones estén correctas.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

CLASS_NAMES  = ['drone', 'person']
CLASS_COLORS = [(0, 255, 100), (255, 80, 80)]  # verde=drone, rojo=person

def draw_yolo_boxes(img_path, lbl_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
                cv2.putText(img, label, (x1, max(y1-5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

# Mostrar 6 imágenes aleatorias del set de entrenamiento
img_files = list((UNIFIED_DIR / 'images' / 'train').glob('*'))[:500]
sample    = random.sample(img_files, min(6, len(img_files)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Muestra del dataset unificado — Verde: drone | Rojo: person', fontsize=14)

for ax, img_path in zip(axes.flat, sample):
    lbl_path = UNIFIED_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    img = draw_yolo_boxes(img_path, lbl_path)
    if img is not None:
        ax.imshow(img)
        ax.set_title(img_path.name[:30], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'dataset_sample.png'), dpi=120, bbox_inches='tight')
plt.show()
print('✅ Muestra guardada en dataset_sample.png')

---
## 🎨 CELDA 7 — Data Augmentation

### Transformaciones aplicadas y justificación:

| Transformación | Parámetro | Justificación para drones |
|---|---|---|
| HorizontalFlip | p=0.5 | Drones pueden aparecer desde cualquier lado |
| RandomRotate90 | p=0.5 | Perspectiva nadir: sin orientación fija |
| Rotate | ±15° | Cabeceo/balanceo suave del dron portador |
| RandomBrightnessContrast | ±20% | Vuelo en distintas horas del día |
| GaussianBlur | p=0.3 | Movimiento rápido del dron → motion blur |
| GaussNoise | p=0.2 | Ruido del sensor de cámara del dron |
| RandomFog | p=0.1 | Condiciones meteorológicas adversas |
| CLAHE | p=0.2 | Mejora contraste en escenas sobreexpuestas |

> **Nota:** YOLOv8 aplica Mosaic augmentation internamente durante el entrenamiento,  
> lo que combina 4 imágenes en una — esto es especialmente útil para objetos pequeños  
> como drones a distancia.

In [ ]:
import albumentations as A
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

# Pipeline de aumentación (se puede aplicar offline para ampliar el dataset)
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(p=0.2),
    A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, p=0.1),
    A.CLAHE(clip_limit=2.0, p=0.2),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


def augment_and_save(img_path, lbl_path, output_img_dir, output_lbl_dir, n_augments=3):
    """Genera n_augments versiones aumentadas de una imagen con sus etiquetas."""
    img = cv2.imread(str(img_path))
    if img is None:
        return 0
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    bboxes, class_labels = [], []
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_labels.append(int(parts[0]))
                    bboxes.append(list(map(float, parts[1:5])))

    if not bboxes:
        return 0

    saved = 0
    for i in range(n_augments):
        try:
            result = augmentation_pipeline(
                image=img, bboxes=bboxes, class_labels=class_labels)
            aug_img = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
            stem = f'aug{i}_{img_path.stem}'
            cv2.imwrite(str(output_img_dir / f'{stem}.jpg'), aug_img)
            with open(output_lbl_dir / f'{stem}.txt', 'w') as f:
                for cls_id, bbox in zip(result['class_labels'], result['bboxes']):
                    f.write(f'{cls_id} {" ".join(map(str, bbox))}\n')
            saved += 1
        except Exception:
            pass
    return saved


# Visualizar efecto de augmentation en 1 imagen de muestra
sample_imgs = list((UNIFIED_DIR / 'images' / 'train').glob('*'))
if sample_imgs:
    sample_img = random.choice(sample_imgs)
    sample_lbl = UNIFIED_DIR / 'labels' / 'train' / (sample_img.stem + '.txt')

    img = cv2.cvtColor(cv2.imread(str(sample_img)), cv2.COLOR_BGR2RGB)
    bboxes, class_labels = [], []
    if sample_lbl.exists():
        with open(sample_lbl) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    class_labels.append(int(p[0]))
                    bboxes.append(list(map(float, p[1:5])))

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('Efecto del Data Augmentation', fontsize=13)
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')

    for idx in range(1, 4):
        try:
            result = augmentation_pipeline(
                image=img, bboxes=bboxes if bboxes else [[0.5,0.5,0.1,0.1]],
                class_labels=class_labels if class_labels else [0])
            axes[idx].imshow(result['image'])
            axes[idx].set_title(f'Augment #{idx}')
        except Exception as e:
            axes[idx].set_title(f'Error: {e}')
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'augmentation_sample.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Muestra de augmentation guardada')

print('\n💡 La aumentación offline se puede ejecutar en la celda siguiente.')
print('   YOLOv8 también aplica Mosaic y otras aumentaciones internamente durante el entrenamiento.')

In [ ]:
# OPCIONAL: Aumentación offline para ampliar el dataset de entrenamiento
# Solo ejecutar si quieres generar imágenes aumentadas físicamente en disco
# Puede tardar varios minutos dependiendo del tamaño del dataset

RUN_OFFLINE_AUGMENTATION = False  # Cambiar a True para ejecutar
N_AUGMENTS_PER_IMAGE     = 2      # Cuántas versiones por imagen original

if RUN_OFFLINE_AUGMENTATION:
    from tqdm import tqdm
    img_files = list((UNIFIED_DIR / 'images' / 'train').glob('*'))
    total_saved = 0
    for img_path in tqdm(img_files, desc='Augmentando imágenes'):
        lbl_path = UNIFIED_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
        total_saved += augment_and_save(
            img_path, lbl_path,
            UNIFIED_DIR / 'images' / 'train',
            UNIFIED_DIR / 'labels' / 'train',
            n_augments=N_AUGMENTS_PER_IMAGE
        )
    print(f'✅ Aumentación completa: {total_saved} imágenes adicionales generadas')
else:
    print('⏭️  Aumentación offline omitida (YOLOv8 aplica augmentation en tiempo real durante el entrenamiento)')

---
## 🏋️ CELDA 8 — Entrenamiento con YOLOv8

### Arquitectura YOLOv8 — Justificación de la elección:

YOLOv8 (You Only Look Once v8) es una red de **detección de objetos en una sola pasada** que divide la imagen en una cuadrícula y predice simultáneamente bounding boxes, confianzas y clases para cada celda. Usa un backbone **CSPDarknet**, un cuello **PANet** para fusión multi-escala, y una cabeza de detección **desacoplada** (separando clasificación de localización).

**¿Por qué YOLOv8 sobre Faster R-CNN o SSD?**
- **Velocidad:** Opera en tiempo real (~60 FPS en RTX 3070), esencial para drones
- **Precisión en objetos pequeños:** La cabeza multi-escala detecta drones a distancia
- **NMS integrado:** Filtra detecciones duplicadas automáticamente
- **Transfer learning:** Preentrenado en COCO con 80 clases → converge rápido

**Variante elegida: YOLOv8m (Medium)**  
- Balance óptimo velocidad/precisión para RTX 3070 con 8GB VRAM
- 25.9M parámetros (vs 3.2M nano, 68.2M large)

### Hiperparámetros y justificación:

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| epochs | 50 | Suficiente para converger con transfer learning |
| imgsz | 640 | Estándar YOLO, balance velocidad/detección |
| batch | 16 | Máximo seguro para 8GB VRAM con yolov8m |
| lr0 | 0.01 | Learning rate inicial (SGD con warmup) |
| lrf | 0.01 | Factor final del scheduler coseno |
| mosaic | 1.0 | Combina 4 imágenes, mejora detección de objetos pequeños |
| augment | True | Aumentaciones en tiempo real de YOLOv8 |
| patience | 15 | Early stopping si no mejora mAP50 |
| optimizer | SGD | Mejor generalización que Adam para detección |

In [ ]:
from ultralytics import YOLO
import torch

# Verificar GPU antes de entrenar
device = '0' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Entrenando en: {"GPU: " + torch.cuda.get_device_name(0) if device == "0" else "CPU (lento)"}')

# Cargar modelo preentrenado YOLOv8m
model = YOLO('yolov8m.pt')  # Se descarga automáticamente (~50MB)

# ============================================================
#                    ENTRENAMIENTO
# ============================================================
results = model.train(
    data    = str(yaml_path),
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    device  = device,

    # Hiperparámetros de optimización
    optimizer = 'SGD',
    lr0       = 0.01,      # Learning rate inicial
    lrf       = 0.01,      # Factor LR final (cosine scheduler)
    momentum  = 0.937,
    weight_decay = 0.0005,

    # Aumentación de datos en tiempo real
    mosaic    = 1.0,       # Combina 4 imágenes en mosaico
    mixup     = 0.1,       # Mezcla dos imágenes con peso alpha
    flipud    = 0.3,       # Flip vertical (perspectiva aérea)
    fliplr    = 0.5,       # Flip horizontal
    degrees   = 10.0,      # Rotación aleatoria ±10°
    translate = 0.1,       # Traslación ±10%
    scale     = 0.5,       # Zoom aleatorio
    hsv_h     = 0.015,     # Variación de tono
    hsv_s     = 0.7,       # Variación de saturación
    hsv_v     = 0.4,       # Variación de brillo

    # Control del entrenamiento
    patience  = 15,        # Early stopping
    save      = True,
    save_period = 10,      # Guardar checkpoint cada 10 épocas
    project   = str(BASE_DIR / 'runs'),
    name      = 'drone_person_yolov8m',
    exist_ok  = True,
    verbose   = True,
    plots     = True,      # Genera curvas automáticamente
)

print('\n✅ Entrenamiento completado')
print(f'📁 Resultados en: {BASE_DIR}/runs/drone_person_yolov8m/')

---
## 📊 CELDA 9 — Métricas y curvas de entrenamiento

### Interpretación de métricas:
- **Box Loss:** Error de localización del bounding box (cuanto menor, más preciso el encuadre)
- **Cls Loss:** Error de clasificación (cuanto menor, mejor distingue drone vs person)
- **DFL Loss:** Distribution Focal Loss — penaliza distribución de bordes del box
- **Precision:** De todas las detecciones positivas, ¿cuántas son correctas? → TP/(TP+FP)
- **Recall:** De todos los objetos reales, ¿cuántos detectó? → TP/(TP+FN)
- **mAP@50:** Mean Average Precision con IoU=0.5 — métrica principal de detección
- **mAP@50-95:** mAP promediado de IoU=0.5 a 0.95 — más exigente, evalúa precisión del box

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

results_dir = BASE_DIR / 'runs' / 'drone_person_yolov8m'
csv_path    = results_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print('Columnas disponibles:', df.columns.tolist())

    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('Curvas de Entrenamiento — YOLOv8m | Drone + Person Detection', fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 4, figure=fig)

    epochs = df['epoch'] + 1

    # Loss curves
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs, df['train/box_loss'], label='Train', color='royalblue')
    ax1.plot(epochs, df['val/box_loss'],   label='Val',   color='tomato', linestyle='--')
    ax1.set_title('Box Loss')
    ax1.set_xlabel('Época'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs, df['train/cls_loss'], color='royalblue', label='Train')
    ax2.plot(epochs, df['val/cls_loss'],   color='tomato', linestyle='--', label='Val')
    ax2.set_title('Classification Loss')
    ax2.set_xlabel('Época'); ax2.legend(); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(epochs, df['train/dfl_loss'], color='royalblue', label='Train')
    ax3.plot(epochs, df['val/dfl_loss'],   color='tomato', linestyle='--', label='Val')
    ax3.set_title('DFL Loss')
    ax3.set_xlabel('Época'); ax3.legend(); ax3.grid(alpha=0.3)

    # Métricas
    ax4 = fig.add_subplot(gs[0, 3])
    ax4.plot(epochs, df['metrics/precision(B)'], color='green',  label='Precision')
    ax4.plot(epochs, df['metrics/recall(B)'],    color='orange', label='Recall')
    ax4.set_title('Precision & Recall')
    ax4.set_xlabel('Época'); ax4.set_ylim(0, 1.05)
    ax4.legend(); ax4.grid(alpha=0.3)

    ax5 = fig.add_subplot(gs[1, 0:2])
    ax5.plot(epochs, df['metrics/mAP50(B)'],    color='purple',  linewidth=2, label='mAP@50')
    ax5.plot(epochs, df['metrics/mAP50-95(B)'], color='magenta', linewidth=2, label='mAP@50-95', linestyle='--')
    ax5.set_title('mAP — Métrica principal de detección')
    ax5.set_xlabel('Época'); ax5.set_ylim(0, 1.05)
    ax5.legend(); ax5.grid(alpha=0.3)

    ax6 = fig.add_subplot(gs[1, 2:4])
    ax6.plot(epochs, df['lr/pg0'], color='gray', label='LR grupo 0')
    ax6.set_title('Learning Rate (Cosine Scheduler)')
    ax6.set_xlabel('Época'); ax6.legend(); ax6.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(BASE_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Resumen numérico final
    last = df.iloc[-1]
    best_map = df['metrics/mAP50(B)'].max()
    best_ep  = df['metrics/mAP50(B)'].idxmax() + 1
    print('\n' + '='*55)
    print('         RESUMEN FINAL DEL ENTRENAMIENTO')
    print('='*55)
    print(f'  mAP@50 mejor        : {best_map:.4f}  (época {best_ep})')
    print(f'  mAP@50-95 final     : {last["metrics/mAP50-95(B)"]:.4f}')
    print(f'  Precision final     : {last["metrics/precision(B)"]:.4f}')
    print(f'  Recall final        : {last["metrics/recall(B)"]:.4f}')
    print(f'  Val Box Loss final  : {last["val/box_loss"]:.4f}')
    print(f'  Val Cls Loss final  : {last["val/cls_loss"]:.4f}')
    print('='*55)
else:
    print('❌ Archivo results.csv no encontrado.')
    print(f'   Verifica que el entrenamiento completó en: {results_dir}')

---
## 🔬 CELDA 10 — Evaluación en conjunto de prueba

In [ ]:
from ultralytics import YOLO

# Cargar el mejor modelo guardado
best_model_path = BASE_DIR / 'runs' / 'drone_person_yolov8m' / 'weights' / 'best.pt'

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))

    # Evaluación en el conjunto de TEST
    test_results = best_model.val(
        data   = str(yaml_path),
        split  = 'test',
        imgsz  = 640,
        device = device,
        conf   = 0.25,
        iou    = 0.5,
    )

    print('\n📊 MÉTRICAS EN CONJUNTO DE PRUEBA:')
    print(f'  mAP@50      : {test_results.box.map50:.4f}')
    print(f'  mAP@50-95   : {test_results.box.map:.4f}')
    print(f'  Precision   : {test_results.box.mp:.4f}')
    print(f'  Recall      : {test_results.box.mr:.4f}')
    print('\nPor clase:')
    for i, cls_name in enumerate(['drone', 'person']):
        print(f'  {cls_name:8s}: AP50 = {test_results.box.ap50[i]:.4f}')
else:
    print(f'❌ Modelo no encontrado en: {best_model_path}')
    print('   Asegúrate de haber completado el entrenamiento primero.')

---
## 🔍 CELDA 11 — Análisis visual de predicciones (éxitos y fallos)

Verde = ground truth (real) | Rojo = predicción del modelo

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
from pathlib import Path
from ultralytics import YOLO

best_model = YOLO(str(BASE_DIR / 'runs' / 'drone_person_yolov8m' / 'weights' / 'best.pt'))

CLASS_NAMES = ['drone', 'person']

def iou(box_a, box_b):
    """Calcula Intersection over Union entre dos boxes (x1y1x2y2)."""
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0

def analyze_image(img_path, lbl_path, model, conf_thresh=0.25):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    display = img.copy()

    # Ground truth (verde)
    gt_boxes = []
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1,y1 = int((cx-bw/2)*w), int((cy-bh/2)*h)
                x2,y2 = int((cx+bw/2)*w), int((cy+bh/2)*h)
                gt_boxes.append((cls_id, x1, y1, x2, y2))
                cv2.rectangle(display, (x1,y1), (x2,y2), (0,200,0), 2)
                cv2.putText(display, f'GT:{CLASS_NAMES[cls_id]}', (x1, max(y1-6,12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,200,0), 2)

    # Predicciones (rojo)
    preds = model(img_path, conf=conf_thresh, verbose=False)[0]
    pred_boxes = []
    for box in preds.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        pred_boxes.append((cls_id, x1, y1, x2, y2, conf))
        cv2.rectangle(display, (x1,y1), (x2,y2), (220,40,40), 2)
        cv2.putText(display, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                    (x1, min(y2+18, h-2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220,40,40), 2)

    # Calcular IoU promedio (calidad de la predicción)
    ious = []
    for gt in gt_boxes:
        for pred in pred_boxes:
            if gt[0] == pred[0]:  # misma clase
                ious.append(iou(gt[1:5], pred[1:5]))
    avg_iou = np.mean(ious) if ious else 0.0

    return display, len(gt_boxes), len(pred_boxes), avg_iou


# Seleccionar imágenes de test
test_imgs = list((UNIFIED_DIR / 'images' / 'test').glob('*'))
random.shuffle(test_imgs)

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
fig.suptitle('Análisis Visual — Verde: Ground Truth | Rojo: Predicción', fontsize=13, fontweight='bold')

results_info = []
for ax, img_path in zip(axes.flat, test_imgs[:6]):
    lbl_path = UNIFIED_DIR / 'labels' / 'test' / (img_path.stem + '.txt')
    display, n_gt, n_pred, avg_iou = analyze_image(img_path, lbl_path, best_model)
    results_info.append((img_path.name, n_gt, n_pred, avg_iou))

    quality = '✅ Éxito' if avg_iou > 0.5 else ('⚠️ Parcial' if avg_iou > 0.2 else '❌ Fallo')
    ax.imshow(display)
    ax.set_title(f'{quality} | GT:{n_gt} Pred:{n_pred} | IoU:{avg_iou:.2f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(BASE_DIR / 'visual_analysis.png'), dpi=130, bbox_inches='tight')
plt.show()

print('\n📋 RESUMEN POR IMAGEN:')
print(f'{"Imagen":<35} {"GT":>4} {"Pred":>5} {"IoU":>6} {"Resultado"}')
print('-' * 65)
for name, n_gt, n_pred, avg_iou in results_info:
    quality = 'Éxito' if avg_iou > 0.5 else ('Parcial' if avg_iou > 0.2 else 'Fallo')
    print(f'{name[:35]:<35} {n_gt:>4} {n_pred:>5} {avg_iou:>6.3f} {quality}')

---
## 🎮 CELDA 12 — Demo de Simulación

Elige el modo de inferencia: imagen estática, video, o cámara web en tiempo real.  
Cambia `MODE` según tu caso de uso.

In [ ]:
import cv2
import time
import numpy as np
from ultralytics import YOLO
from pathlib import Path

# ============================================================
# CONFIGURACIÓN DE LA SIMULACIÓN
# ============================================================
MODE         = 'webcam'        # 'image' | 'video' | 'webcam'
INPUT_PATH   = 'test_video.mp4'  # Solo para MODE='image' o 'video'
CONF_THRESH  = 0.35            # Umbral de confianza
IOU_THRESH   = 0.45            # Umbral IoU para NMS
SAVE_OUTPUT  = True            # Guardar video de salida
# ============================================================

best_model = YOLO(str(BASE_DIR / 'runs' / 'drone_person_yolov8m' / 'weights' / 'best.pt'))

CLASS_COLORS = {0: (0, 255, 120), 1: (60, 130, 255)}  # drone=verde, person=azul
CLASS_NAMES  = ['drone', 'person']

def draw_detections(frame, results, elapsed_ms):
    """Dibuja bounding boxes, etiquetas, FPS y estadísticas."""
    h, w = frame.shape[:2]
    counts = {0: 0, 1: 0}

    for box in results[0].boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        color  = CLASS_COLORS.get(cls_id, (200,200,200))

        # Bounding box con esquinas decorativas
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        corner_len = min(20, (x2-x1)//4, (y2-y1)//4)
        for (cx,cy) in [(x1,y1),(x2,y1),(x1,y2),(x2,y2)]:
            dx = corner_len if cx == x1 else -corner_len
            dy = corner_len if cy == y1 else -corner_len
            cv2.line(frame, (cx,cy), (cx+dx,cy), color, 3)
            cv2.line(frame, (cx,cy), (cx,cy+dy), color, 3)

        label = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
        cv2.putText(frame, label, (x1+2, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
        counts[cls_id] = counts.get(cls_id, 0) + 1

    # HUD de información
    fps = 1000 / elapsed_ms if elapsed_ms > 0 else 0
    overlay = frame.copy()
    cv2.rectangle(overlay, (0,0), (240, 75), (20,20,20), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, f'FPS: {fps:.1f}', (8,22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
    cv2.putText(frame, f'Drones: {counts[0]}', (8,46),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, CLASS_COLORS[0], 2)
    cv2.putText(frame, f'Persons: {counts[1]}', (8,70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, CLASS_COLORS[1], 2)

    if counts[0] > 0:
        cv2.putText(frame, '⚠ DRONE NEARBY', (w//2-120, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0,60,255), 3)
    return frame


# --- Imagen estática ---
if MODE == 'image':
    import matplotlib.pyplot as plt
    t0 = time.time()
    results = best_model(INPUT_PATH, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)
    elapsed = (time.time()-t0)*1000
    frame = cv2.cvtColor(cv2.imread(INPUT_PATH), cv2.COLOR_BGR2RGB)
    frame = draw_detections(frame, results, elapsed)
    plt.figure(figsize=(14,8))
    plt.imshow(frame); plt.axis('off')
    plt.title(f'Detección en imagen — {len(results[0].boxes)} objetos detectados')
    plt.savefig(str(BASE_DIR / 'inference_image.png'), dpi=130, bbox_inches='tight')
    plt.show()

# --- Video o Webcam ---
elif MODE in ('video', 'webcam'):
    source  = 0 if MODE == 'webcam' else INPUT_PATH
    cap     = cv2.VideoCapture(source)
    fw, fh  = int(cap.get(3)), int(cap.get(4))
    writer  = None

    if SAVE_OUTPUT:
        out_path = str(BASE_DIR / f'inference_{MODE}.mp4')
        writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 20, (fw, fh))
        print(f'📹 Guardando en: {out_path}')

    print(f'🎮 Simulación activa [{MODE.upper()}] — Presiona Q para salir')

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        t0      = time.time()
        results = best_model(frame, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)
        elapsed = (time.time()-t0)*1000
        frame   = draw_detections(frame, results, elapsed)

        if writer:
            writer.write(frame)

        cv2.imshow('Drone & Person Detection | Presiona Q para salir', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print('Simulación detenida por el usuario')
            break

    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()
    print('✅ Simulación finalizada')

---
## 💾 CELDA 13 — Exportar modelo para despliegue en dron real

In [ ]:
from ultralytics import YOLO

best_model = YOLO(str(BASE_DIR / 'runs' / 'drone_person_yolov8m' / 'weights' / 'best.pt'))

# Exportar a ONNX (compatible con la mayoría de plataformas de drones)
export_path = best_model.export(format='onnx', imgsz=640, optimize=True)
print(f'✅ Modelo ONNX exportado: {export_path}')

# Información del modelo
info = best_model.info()
print(f'\n📋 Parámetros del modelo: {info}')
print('\n📦 Archivos para despliegue:')
print(f'  Modelo PyTorch  : {BASE_DIR}/runs/drone_person_yolov8m/weights/best.pt')
print(f'  Modelo ONNX     : {export_path}')
print('\n💡 Para drones Jetson Nano/Xavier, exporta con: format="engine" (TensorRT)')

---
## 📝 Notas para el Reporte Técnico

### Justificación de Non-Maximum Suppression (NMS)
YOLOv8 aplica NMS automáticamente después de la inferencia. El parámetro `iou=0.45` controla el umbral: dos detecciones que se solapan más del 45% → se elimina la de menor confianza. Esto es crítico en enjambres de drones donde múltiples celdas de la cuadrícula pueden detectar el mismo dron.

### Interpretación de fallos esperados
- **Falsos negativos en drones lejanos:** El dron es muy pequeño en pixeles, el modelo pierde resolución
- **Confusión drone/pájaro:** Silueta similar en fondos de cielo → aumentación con fondos de cielo ayuda
- **Personas parcialmente ocluidas:** Dataset de entrenamiento tiene pocas imágenes de oclusión parcial

### Métricas esperadas con YOLOv8m y ~10K imágenes
- mAP@50 > 0.75 para clase drone
- mAP@50 > 0.80 para clase person (más imágenes de entrenamiento)
- FPS en RTX 3070: ~80-120 FPS en inferencia